# Meta - Instagram Creative Post Engagement and Sharing Patterns

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [5]:
df_agg = pd.read_csv('../Data/022/agg_daily_creative_shares.csv', parse_dates=['share_date'])

pl_agg = pl.read_csv('../Data/022/agg_daily_creative_shares.csv').with_columns(pl.col('share_date').str.to_date("%Y-%m-%d"))

# Pregunta 1

### ¿Qué usuarios compartieron fotos o videos creativos (es decir, la suma total de veces que compartieron) más de 10 veces en abril de 2024? Este análisis ayudará a determinar qué usuarios están altamente comprometidos con la compartición de contenido.

```SQL
SELECT
    user_id
FROM agg_daily_creative_shares
WHERE ((EXTRACT(MONTH FROM share_date) = 4) AND
       (EXTRACT(YEAR FROM share_date) = 2024))
GROUP BY user_id
HAVING SUM(share_count) > 10;
```

In [8]:
abril = df_agg[
    (df_agg['share_date'].dt.month == 4) &
    (df_agg['share_date'].dt.year == 2024)
].reset_index()

res = abril.groupby('user_id')['share_count'].sum().reset_index()

res = res[res['share_count'] > 10]

res = res[['user_id']]

In [11]:
res = pl_agg.filter(
    (pl.col('share_date').dt.month() == 4) &
    (pl.col('share_date').dt.year() == 2024)
).group_by('user_id').agg(
    pl.col('share_count').sum().alias('total_shares')
).filter(
    pl.col('total_shares') > 10
).select(
    pl.col('user_id')
)

res

user_id
i64
106
101
104
103


# Pregunta 2

### ¿Cuál es el promedio de contenidos compartidos por usuario en mayo de 2024, entre aquellos que compartieron al menos una vez? Primero queremos obtener el total de compartidos por cada usuario en mayo de 2024 y, después, calcular el promedio sobre todos esos usuarios.

```SQL
SELECT
    ROUND(AVG(total_por_usuario),2) AS promedio_final
FROM(
SELECT
    user_id,
    SUM(share_count) AS total_por_usuario
FROM agg_daily_creative_shares
WHERE ((EXTRACT(MONTH FROM share_date) = 5) AND
       (EXTRACT(YEAR FROM share_date) = 2024))
GROUP BY user_id
) AS sub_query

In [20]:
mayo = df_agg[
    (df_agg['share_date'].dt.month == 5) &
    (df_agg['share_date'].dt.year == 2024)
].reset_index()

res = mayo.groupby('user_id')['share_count'].sum().reset_index(name='total_por_usuario')

res_final = res['total_por_usuario'].mean().round(2)

res_final

np.float64(7.83)

In [22]:
res = pl_agg.filter(
    (pl.col('share_date').dt.month() == 5) &
    (pl.col('share_date').dt.year() == 2024)
).group_by('user_id').agg(
    pl.col('share_count').sum().alias('total_por_usuario')
).select(
    pl.col('total_por_usuario').mean().round(2).alias('promedio')
)

res

promedio
f64
7.83


# Pregunta 3

### Para cada usuario de Instagram que compartió contenido creativo, ¿cuál es el valor redondeado hacia abajo (floor) de su promedio de compartidos diarios durante el segundo trimestre de 2024? Incluye únicamente a los usuarios con un promedio de al menos 5 compartidos por día.

### Nota: la tabla agg_daily_creative_shares está al nivel de tipo de contenido, usuario y día. Asegúrate de agregar los datos al nivel de 'usuario-día' antes de calcular el promedio.

```SQL
SELECT
    user_id,
    FLOOR(AVG(total_diario)) AS avg_daily_shares
FROM (SELECT user_id,
             SUM(share_count) AS total_diario
      FROM agg_daily_creative_shares
      WHERE share_date BETWEEN '2024-04-01' AND '2024-06-30'
      GROUP BY user_id, share_date) AS tabla_diarai
GROUP BY user_id
HAVING AVG(total_diario) >= 5;
```

In [37]:
df_q2 = df_agg[
    (df_agg['share_date'].between('2024-04-01','2024-06-30'))
].reset_index()

res = df_q2.groupby(['user_id','share_date'])['share_count'].sum().reset_index(name='total_diario')

res_f = res.groupby('user_id')['total_diario'].mean().reset_index(name='avg_daily_shares')

res_f = res_f[res_f['avg_daily_shares'] >= 5].copy()
res_f['avg_daily_shares'] = np.floor(res_f['avg_daily_shares'])

In [41]:
res = pl_agg.filter(
    (pl.col('share_date').is_between(date(2024,4,1), date(2024,6,30)))
).group_by(['user_id','share_date']).agg(
    pl.col('share_count').sum().alias('total_diario')
).group_by('user_id').agg(
    pl.col('total_diario').mean().floor().alias('avg_daily_shares')
).filter(
    (pl.col('avg_daily_shares') >= 5)
)

res

user_id,avg_daily_shares
i64,f64
101,5.0
103,5.0
106,6.0
